# Parte 2 - Del LLM al clasificador

## 2.1. Línea base Zero-Shot (sin fine-tuning)

**Enunciado**: 
- Evaluar 40 titulares usando pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli") o Recognai/bert-base-spanish-wwm-cased-xnli.
- Reportar accuracy y F1 macro. Este puntaje representa el piso mínimo a superar.

Antes de iniciar, se instaló transformers + torch + accelerate, datasets, scikit-learn, seaborn, google-genai, python-dotenv, y se registró la clave de Gemini, la IA que se usará.

In [1]:
import os
from dotenv import load_dotenv
import transformers, torch

load_dotenv()  # lee el archivo .env
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("¿Clave de Gemini cargada?", os.getenv("GEMINI_API_KEY") is not None)

c:\Users\LENOVO\Documents\GitHub\Python_intermedio\tarea_1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 5.20.1 | torch 2.14.1+cpu
¿Clave de Gemini cargada? True


Se verifica que se cargó correctamente la clave de Gemini.

Ahora, el zero-shot es un modelo ya entrenado para decidir si una frase encaja o no con una etiqueta escrita en palabras (explicado por Claude). 

Para la prueba, como indica la tarea, se tomarán 40 titulares, 10 por cada categoría. 

In [ ]:
import pandas as pd

noticias = pd.read_csv("datos/noticias.csv")

# 10 titulares al azar de cada categoría (semilla fija para que siempre salgan los mismos)
muestra_zs = noticias.groupby("etiqueta").sample(10, random_state=42)
print(muestra_zs.shape)
muestra_zs["etiqueta"].value_counts()

(40, 7)


etiqueta
ciencia_tecnologia    10
deportes              10
economia              10
politica              10
Name: count, dtype: int64

Una vez seleccionados los 40 titulares, se procede a aplicar el modelo zero-shot. Para ello, se determinó que se pruebe con "Esta noticia trata sobre..." política/economía/deportes/ciencia y tecnología.

In [3]:
from transformers import pipeline

clasificador_zs = pipeline("zero-shot-classification",
                           model="Recognai/bert-base-spanish-wwm-cased-xnli")

# El modelo necesita las categorías escritas en palabras; luego se traducen a las etiquetas oficiales
CATEGORIAS = {
    "política": "politica",
    "economía": "economia",
    "deportes": "deportes",
    "ciencia y tecnología": "ciencia_tecnologia",
}

resultados = clasificador_zs(list(muestra_zs["titular"]),
                             candidate_labels=list(CATEGORIAS),
                             hypothesis_template="Esta noticia trata sobre {}.")

# Para cada titular, la categoría con mayor puntaje
muestra_zs["prediccion_zs"] = [CATEGORIAS[r["labels"][0]] for r in resultados]


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 160.87it/s]


Ahora, se calculan las métricas: accuracy (porcentaje de aciertos) y F1 macro (promedio del desempeño en cada categoría).

In [4]:
from sklearn.metrics import accuracy_score, f1_score

acc_zs = accuracy_score(muestra_zs["etiqueta"], muestra_zs["prediccion_zs"])
f1_zs = f1_score(muestra_zs["etiqueta"], muestra_zs["prediccion_zs"], average="macro")
print(f"Zero-shot -> Accuracy: {acc_zs:.3f} | F1 macro: {f1_zs:.3f}")

pd.crosstab(muestra_zs["etiqueta"], muestra_zs["prediccion_zs"])

Zero-shot -> Accuracy: 0.575 | F1 macro: 0.560


prediccion_zs,ciencia_tecnologia,deportes,economia,politica
etiqueta,,,,
ciencia_tecnologia,9,1,0,0
deportes,0,5,0,5
economia,1,1,2,6
politica,0,1,2,7


Los valores de accurancy es de 0,575, y de F1 macro es 0,560, los cuales son bajos.

Se identifica que en ciencia_tecnologia acertó en casi todos (9 de 10), pero en deportes, economía y política, los aciertos bajan (5, 2 y 7 de 10, respectivamente). 

Además, se identifica que el modelo tiende a clasificar las noticias de deporte como político (5 de 10), así como de economía como político (6 de 10).

Frente a ello, se espera que este método base debe ser superado por los otros métodos.